# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.14077628 -0.03323694  0.93859333 -0.48337038 -0.75379381]
 [-0.08156008  0.64780794  0.16046798  0.99219932 -0.12008573]
 [-0.95418127 -0.30782342  0.48969272 -0.5932156   0.83717956]
 [ 0.57963058 -0.21624101  0.61089987 -0.83382741 -0.67690054]
 [ 0.95573154  0.12934648  0.34423694 -0.10895166 -0.62826387]
 [-0.3891387   0.93991835  0.8939779  -0.64980669 -0.90648678]
 [ 0.2862322   0.42507152 -0.91384487 -0.95995997 -0.05605696]
 [-0.61630658  0.36813546 -0.5779627   0.20817694  0.86860938]
 [-0.48822238 -0.9548769   0.09906524 -0.91460788  0.14668372]
 [ 0.73723577  0.61824517 -0.70845412 -0.39989631 -0.96346696]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a1', 'a1', 'a1', 'a2', 'a1', 'a2', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 1, 0, 1, 0, 1, 0, 1, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:33,  1.02s/it]

SVI:   3%|▎         | 1/34 [00:01<00:33,  1.02s/it, loss=1268.7109]

SVI:   6%|▌         | 2/34 [00:01<00:32,  1.02s/it, loss=1288.3146]

SVI:   9%|▉         | 3/34 [00:01<00:31,  1.02s/it, loss=1257.9314]

...

SVI:  84%|████████▍ | 21/25 [00:00<00:03,  1.18it/s, loss=1060.5449]

SVI:  88%|████████▊ | 22/25 [00:00<00:02,  1.18it/s, loss=1183.6761]

SVI:  92%|█████████▏| 23/25 [00:00<00:01,  1.18it/s, loss=1022.1623]

SVI:  96%|█████████▌| 24/25 [00:00<00:00,  1.18it/s, loss=1125.2271]

SVI: 100%|██████████| 25/25 [00:00<00:00,  1.18it/s, loss=1142.2886]